# Cross-task final-position patching — corrected donor target, all within-family pairs

Run a SOURCE prompt; at layer L overwrite the final-position residual with a DONOR prompt's
final-position residual; generate; classify the output with EXACT full-string match:

- **acc_src**   : output == source_rule(source_query_input)   -> still doing the original task
- **acc_donor** : output == donor_rule(SOURCE_query_input)    -> switched to the donor task
- **other**     : neither

Key fix: the donor target is the donor task's RULE applied to the SOURCE's query input
(donor_rule(source_input)), NOT the donor prompt's own stored answer. That makes 'follows
donor' a real measurement.

Runs all ordered pairs WITHIN nonce tasks and WITHIN arithmetic tasks. `experiments/interplay/`.

In [1]:
import os, sys, pickle, itertools
sys.path.insert(0, os.path.abspath('../..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

import experiments.pairing._common as C
from data.loaders import load_dataset
from data.tasks import _NONCE_TRANSFORMS, _ARITH_SPECS, NONCE_TASKS, ARITH_TASKS

## Config + rules

In [2]:
DATASET = 'nonce+arithmetic'
N       = 8          # source prompts per (pair, layer)
MAXNEW  = 12
CUDA    = '0'
ds_tag  = DATASET.replace('+','_')

model  = C.load_model(cuda_visible=CUDA)
splits = load_dataset(DATASET)
nL     = model.cfg.n_layers

# apply a task's rule to an arbitrary query input (string for nonce, str-int for arith)
def apply_rule(task, query_input):
    if task in _NONCE_TRANSFORMS:
        try: return _NONCE_TRANSFORMS[task](str(query_input))
        except Exception: return None
    if task in _ARITH_SPECS:
        fn,_,_ = _ARITH_SPECS[task]
        try: return str(fn(int(query_input)))
        except Exception: return None
    return None

# only tasks present in splits
nonce = [t for t in NONCE_TASKS if t in splits]
arith = [t for t in ARITH_TASKS if t in splits]
print(f'{len(nonce)} nonce tasks, {len(arith)} arith tasks')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.2-3B into HookedTransformer
10 nonce tasks, 10 arith tasks


## Patching machinery (exact full-string match)

In [3]:
@torch.no_grad()
def donor_resid_all_layers(prompt):
    arrow=C.query_arrow_position(model,prompt); toks=model.to_tokens(prompt,prepend_bos=True)
    _,c=model.run_with_cache(toks,names_filter=lambda n:'resid_post' in n)
    out={L:c['resid_post',L][0,arrow].float().cpu().numpy() for L in range(nL)}
    del c; torch.cuda.empty_cache(); return out

@torch.no_grad()
def patched_gen(src_prompt, donor_vec, L, max_new=MAXNEW):
    base = model.to_tokens(src_prompt, prepend_bos=True)
    theta = torch.tensor(donor_vec, device=base.device, dtype=model.cfg.dtype)
    inj = base.shape[1]-1
    def hook(v, hook): v[0, inj, :] = theta; return v
    cur=base.clone(); gen=[]
    for _ in range(max_new):
        logits = model.run_with_hooks(cur, fwd_hooks=[(f'blocks.{L}.hook_resid_post',hook)])[0,-1]
        nt=logits.argmax().item(); gen.append(nt)
        cur=torch.cat([cur, torch.tensor([[nt]],device=cur.device)],1)
        dec=model.tokenizer.decode(gen).strip()
        # early stop once long enough to decide either target (efficiency)
        if len(dec) > 20: break
    return model.tokenizer.decode(gen).strip()

def exact(dec, ans):
    return dec.strip() == str(ans).strip() if ans is not None else False

## Run all within-family pairs
For each (src, donor) ordered pair, sweep layers; classify each generation as src / donor / other
by EXACT match. Averaged over N source prompts.

In [ ]:
def run_family(tasks, family):
    rows=[]
    pairs = [(s,d) for s,d in itertools.permutations(tasks, 2)]
    for src_t, dnr_t in tqdm(pairs, desc=family):
        src_ps = splits[src_t]['icl_prompts'][:N]
        # precompute donor residuals from donor prompts (one per source index, cycled)
        dnr_ps = splits[dnr_t]['icl_prompts'][:N]
        dres = [donor_resid_all_layers(dp['prompt']) for dp in dnr_ps]
        # targets per source prompt
        src_inputs = [sp['query_input'] for sp in src_ps]
        src_tgt = [apply_rule(src_t, qi) for qi in src_inputs]   # = sp['query_output']
        dnr_tgt = [apply_rule(dnr_t, qi) for qi in src_inputs]   # donor rule on SOURCE input
        for L in range(nL):
            a_src=a_dnr=oth=0
            for j,sp in enumerate(src_ps):
                dec = patched_gen(sp['prompt'], dres[j][L], L)
                if   exact(dec, src_tgt[j]): a_src+=1
                elif exact(dec, dnr_tgt[j]): a_dnr+=1
                else:                        oth+=1
            n=len(src_ps)
            rows.append({'family':family,'pair':f'{src_t}->{dnr_t}','src':src_t,'donor':dnr_t,
                         'layer':L,'acc_src':a_src/n,'acc_donor':a_dnr/n,'other':oth/n})
    return pd.DataFrame(rows)

R_nonce = run_family(nonce, 'nonce')
R_arith = run_family(arith, 'arith')
R = pd.concat([R_nonce, R_arith], ignore_index=True)
R.to_csv('crosstask_patch_allpairs.csv', index=False)
print('done', R.shape)

nonce:   0%|          | 0/90 [00:00<?, ?it/s]

## Family-average curves (mean over all pairs in each family)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5),squeeze=False)
for ax,(fam,Rf) in zip(axes[0], [('nonce',R_nonce),('arith',R_arith)]):
    g = Rf.groupby('layer')[['acc_src','acc_donor','other']].mean()
    ax.plot(g.index, g['acc_src'],   'o-', label='acc_src (keeps original)', ms=3)
    ax.plot(g.index, g['acc_donor'], 's-', label='acc_donor (switched task)', ms=3)
    ax.plot(g.index, g['other'],     '^-', label='other', ms=3, alpha=.6)
    ax.set(title=f'{fam}: mean over all within-family pairs', xlabel='layer (final-pos patched)',
           ylabel='fraction', ylim=(-.02,1.02))
    ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('crosstask_patch_family_means.png', dpi=130, bbox_inches='tight'); plt.show()

## Peak donor-transfer per pair (does ANY layer install the donor task?)

In [ ]:
# for each pair, the max acc_donor across layers = best evidence the patch installs donor behavior
peak = R.groupby(['family','pair'])['acc_donor'].max().reset_index()
print('mean peak acc_donor by family:')
print(peak.groupby('family')['acc_donor'].mean().round(3).to_string())
print('\ntop pairs by donor-transfer:')
print(peak.sort_values('acc_donor', ascending=False).head(12).to_string(index=False))

fig,ax=plt.subplots(figsize=(7,4.5))
import seaborn as sns
sns.boxplot(peak, x='family', y='acc_donor', ax=ax)
sns.stripplot(peak, x='family', y='acc_donor', color='k', size=3, alpha=.5, ax=ax)
ax.set(title='peak donor-transfer per pair (does patching install the other task?)',
       ylabel='max acc_donor across layers')
plt.tight_layout(); plt.show()

## Read it
- **acc_donor rises at some layer** -> patching the final-position residual INSTALLS the donor
  task (model computes donor_rule on the source input). Genuine task transfer.
- **acc_donor stays ~0, acc_src drops into 'other'** -> patching BREAKS the source task but does
  NOT install the donor: the final-position residual is necessary but not a transplantable task code.
- **acc_src stays ~1 (arith)** -> final-position residual isn't even necessary; task computed robustly elsewhere.
- Compare nonce vs arith family means, and the peak-donor boxplot: if nonce shows donor-transfer
  at mid layers but arith shows none, that's a clean task-family dissociation in transferability.